# Master Game-Level Dataset: All MLB Regular Season Games (2015-2025)

This notebook builds a centralized dataset of **game-level baseball statistics** for all 30 MLB teams across the 2015-2025 seasons. Each row represents one regular season game.

**Data sources:**
1. **Statcast pitch-by-pitch data** (pybaseball) — aggregated to game level
2. **MLB Stats API** — game start times in UTC

**Purpose:** Avoid redundant Statcast pulls when building stadium-specific datasets. Stadium notebooks read from `master_data.csv` and add their own weather/wind data.

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

## Section 0: Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# pybaseball
from pybaseball import statcast
from pybaseball import cache
cache.enable()  # Cache Statcast data locally to avoid re-downloading

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

SEASONS = range(2015, 2026)  # 2015 through 2025
print("Setup complete. Seasons:", list(SEASONS))

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Setup complete. Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Section 1: Pull Statcast Pitch-Level Data (League-Wide)

Pull pitch-by-pitch Statcast data month-by-month for **all teams** (no team filter). This is the same data each stadium notebook would pull individually — doing it once here avoids redundant downloads.

**Note:** The first run will take 2-4 hours. Subsequent runs use pybaseball's local cache and are near-instant.

In [7]:
# Pull Statcast data month-by-month to avoid parser errors from large requests
# Baseball Savant can return malformed data on big date ranges; smaller chunks are more reliable

all_pitches = []

MONTH_RANGES = [
    ('03-01', '03-31'), ('04-01', '04-30'), ('05-01', '05-31'),
    ('06-01', '06-30'), ('07-01', '07-31'), ('08-01', '08-31'),
    ('09-01', '09-30'), ('10-01', '10-31'), ('11-01', '11-30'),
]

for year in SEASONS:
    year_pitches = 0
    print(f"Pulling {year} season (all teams)...")
    
    for m_start, m_end in MONTH_RANGES:
        start = f"{year}-{m_start}"
        end = f"{year}-{m_end}"
        
        for attempt in range(5):  # Up to 5 retries per month (larger payloads)
            try:
                df = statcast(start_dt=start, end_dt=end, verbose=False)
                if df is not None and len(df) > 0:
                    all_pitches.append(df)
                    year_pitches += len(df)
                break  # Success — exit retry loop
            except Exception as e:
                if attempt < 4:
                    print(f"  Retry {attempt+1} for {start} to {end}: {e}")
                    time.sleep(5)
                else:
                    print(f"  FAILED {start} to {end} after 5 attempts: {e}")
    
    print(f"  {year}: {year_pitches:,} pitches")

pitches_raw = pd.concat(all_pitches, ignore_index=True)
print(f"\nTotal raw pitches (all teams, all game types): {len(pitches_raw):,}")

Pulling 2015 season (all teams)...


0it [00:00, ?it/s]
100%|██████████| 1/1 [00:00<00:00,  1.52it/s]


  2015: 712,844 pitches
Pulling 2016 season (all teams)...


0it [00:00, ?it/s]
 55%|█████▍    | 17/31 [00:50<00:41,  2.95s/it]


  Retry 1 for 2016-07-01 to 2016-07-31: Error: Query Timeout. Please try to limit your query to less data.


...

  FAILED 2025-03-01 to 2025-03-31 after 5 attempts: Error: Query Timeout. Please try to limit your query to less data.


 73%|███████▎  | 11/15 [00:24<00:08,  2.20s/it]


  Retry 1 for 2025-11-01 to 2025-11-30: HTTPSConnectionPool(host='baseballsavant.mlb.com', port=443): Max retries exceeded with url: /statcast_search/csv?all=true&hfPT=&hfAB=&hfBBT=&hfPR=&hfZ=&stadium=&hfBBL=&hfNewZones=&hfGT=R%7CPO%7CS%7C=&hfSea=&hfSit=&player_type=pitcher&hfOuts=&opponent=&pitcher_throws=&batter_stands=&hfSA=&game_date_gt=2025-11-15&game_date_lt=2025-11-15&team=&position=&hfRO=&home_road=&hfFlag=&metric_1=&hfInn=&min_pitches=0&min_results=0&group_by=name&sort_col=pitches&player_event_sort=h_launch_speed&sort_order=desc&min_abs=0&type=details& (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f8545999a00>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))


100%|██████████| 15/15 [00:22<00:00,  1.53s/it]


  2025: 706,987 pitches

Total raw pitches (all teams, all game types): 7,496,670


In [8]:
# Filter to regular season games only
pitches = pitches_raw[pitches_raw['game_type'] == 'R'].copy()

pitches['game_date'] = pd.to_datetime(pitches['game_date'])
pitches['season'] = pitches['game_date'].dt.year

print(f"Filtered pitches (regular season, all teams): {len(pitches):,}")
print(f"Unique games: {pitches['game_pk'].nunique()}")
print(f"Unique home teams: {pitches['home_team'].nunique()}")
print(f"Seasons: {sorted(pitches['season'].unique())}")
print(f"\nGames per season:")
print(pitches.groupby('season')['game_pk'].nunique())

Filtered pitches (regular season, all teams): 7,175,512
Unique games: 24324
Unique home teams: 30
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Games per season:
season
2015    2429
2016    2047
2017    2009
2018    2431
2019    2429
2020     898
2021    2429
2022    2430
2023    2430
2024    2429
2025    2363
Name: game_pk, dtype: int64


## Section 2: Aggregate Pitch Data to Game Level

Compute game-level statistics from pitch-by-pitch data. All stats are **both teams combined**.

In [13]:
# Define event categories
HIT_EVENTS = {'single', 'double', 'triple', 'home_run'}
BATTED_BALL_EVENTS = HIT_EVENTS | {
    'field_out', 'grounded_into_double_play', 'force_out',
    'fielders_choice_out', 'sac_fly', 'field_error',
    'double_play', 'sac_bunt', 'fielders_choice',
    'sac_fly_double_play', 'triple_play'
}

def is_barrel(ev, la):
    """
    MLB barrel definition:
    - Minimum 98 mph exit velocity
    - At 98 mph: launch angle 26-30 degrees
    - For each mph above 98, the LA range expands
    - At 116+ mph: launch angle 8-50 degrees
    """
    if pd.isna(ev) or pd.isna(la):
        return False
    if ev < 98:
        return False
    excess_mph = min(ev - 98, 18)
    la_min = 26 - (excess_mph * 1.0)
    la_max = 30 + (excess_mph * 1.111)
    return la_min <= la <= la_max


def aggregate_game(game_df):
    """Aggregate pitch-level data to a single game row."""
    game_pk = game_df['game_pk'].iloc[0]
    game_date = game_df['game_date'].iloc[0]
    season = game_df['season'].iloc[0]
    home_team = game_df['home_team'].iloc[0]
    away_team = game_df['away_team'].iloc[0]

    # --- RUNS: final score from last pitch of the game ---
    last_pitch = game_df.sort_values(
        ['inning', 'at_bat_number', 'pitch_number'], ascending=True
    ).iloc[-1]
    home_runs_scored = last_pitch['post_home_score'] if pd.notna(last_pitch.get('post_home_score')) else np.nan
    away_runs_scored = last_pitch['post_away_score'] if pd.notna(last_pitch.get('post_away_score')) else np.nan
    total_runs = home_runs_scored + away_runs_scored if pd.notna(home_runs_scored) and pd.notna(away_runs_scored) else np.nan

    # --- EVENTS (plate appearance results only) ---
    pa_df = game_df[game_df['events'].notna()]
    home_runs_hit = (pa_df['events'] == 'home_run').sum()
    strikeouts = pa_df['events'].isin(['strikeout', 'strikeout_double_play']).sum()
    walks = pa_df['events'].isin(['walk', 'intent_walk']).sum()
    hits = pa_df['events'].isin(HIT_EVENTS).sum()

    # --- TOTAL PITCHES ---
    total_pitches = len(game_df)

    # --- BATTED BALL METRICS ---
    batted = game_df[game_df['launch_speed'].notna()]
    avg_exit_velocity = batted['launch_speed'].mean() if len(batted) > 0 else np.nan

    # Barrel rate: barrels / batted ball events
    bbe_df = pa_df[pa_df['events'].isin(BATTED_BALL_EVENTS)]
    n_bbe = len(bbe_df)
    if n_bbe > 0:
        n_barrels = sum(is_barrel(row['launch_speed'], row['launch_angle']) for _, row in bbe_df.iterrows())
        barrel_rate = n_barrels / n_bbe
    else:
        n_barrels = 0
        barrel_rate = np.nan

    # HR / H ratio
    hr_h_ratio = home_runs_hit / hits if hits > 0 else np.nan

    return pd.Series({
        'game_pk': game_pk,
        'game_date': game_date,
        'season': season,
        'home_team': home_team,
        'away_team': away_team,
        'home_runs_scored': home_runs_scored,
        'away_runs_scored': away_runs_scored,
        'total_runs': total_runs,
        'home_runs_hit': home_runs_hit,
        'strikeouts': strikeouts,
        'walks': walks,
        'hits': hits,
        'total_pitches': total_pitches,
        'avg_exit_velocity': avg_exit_velocity,
        'n_barrels': n_barrels,
        'n_bbe': n_bbe,
        'barrel_rate': barrel_rate,
        'hr_h_ratio': hr_h_ratio,
    })

print("Event categories and aggregate_game() defined.")

Event categories and aggregate_game() defined.


In [10]:
print("Aggregating pitch data to game level (all teams)...")
print("This may take several minutes for ~26,000 games...")
games = pitches.groupby('game_pk').apply(aggregate_game).reset_index(drop=True)
games['game_date'] = pd.to_datetime(games['game_date'])

# Ensure integer types for count columns
int_cols = ['game_pk', 'home_runs_scored', 'away_runs_scored', 'total_runs',
            'home_runs_hit', 'strikeouts', 'walks', 'hits', 'total_pitches',
            'n_barrels', 'n_bbe', 'season']
for col in int_cols:
    games[col] = pd.to_numeric(games[col], errors='coerce').astype('Int64')

# Round floating point columns
games['avg_exit_velocity'] = games['avg_exit_velocity'].round(1)
games['barrel_rate'] = games['barrel_rate'].round(4)
games['hr_h_ratio'] = games['hr_h_ratio'].round(4)

games = games.sort_values(['game_date', 'game_pk']).reset_index(drop=True)

print(f"\nTotal games: {len(games)}")
print(f"Unique home teams: {games['home_team'].nunique()}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Aggregating pitch data to game level (all teams)...
This may take several minutes for ~26,000 games...

Total games: 24324
Unique home teams: 30

Games per season:
season
2015    2429
2016    2047
2017    2009
2018    2431
2019    2429
2020     898
2021    2429
2022    2430
2023    2430
2024    2429
2025    2363
Name: game_pk, dtype: Int64

Sample:
   game_pk  game_date  season home_team away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  \
0   413661 2015-04-05    2015       CHC       STL                 0                 3           3              0          23      6    15            314               85.3          1     45   
1   413649 2015-04-06    2015       SEA       LAA                 4                 1           5              2          15      3    14            253               89.9          1     46   
2   413650 2015-04-06    2015       MIA       ATL                 1   

## Section 3: Get Game Start Times from MLB API

Pull game schedules for all 30 teams from the MLB Stats API. Store start times in UTC so each stadium notebook can convert to its own local timezone.

In [15]:
# All 30 MLB team IDs
MLB_TEAM_IDS = {
    'ARI': 109, 'ATL': 144, 'BAL': 110, 'BOS': 111, 'CHC': 112,
    'CIN': 113, 'CLE': 114, 'COL': 115, 'CWS': 145, 'DET': 116,
    'HOU': 117, 'KC': 118, 'LAA': 108, 'LAD': 119, 'MIA': 146,
    'MIL': 158, 'MIN': 142, 'NYM': 121, 'NYY': 147, 'OAK': 133,
    'PHI': 143, 'PIT': 134, 'SD': 135, 'SEA': 136, 'SF': 137,
    'STL': 138, 'TB': 139, 'TEX': 140, 'TOR': 141, 'WSH': 120,
}

MLB_API_URL = "https://statsapi.mlb.com/api/v1/schedule"

print(f"MLB_TEAM_IDS defined ({len(MLB_TEAM_IDS)} teams)")
print(f"MLB_API_URL: {MLB_API_URL}")

MLB_TEAM_IDS defined (30 teams)
MLB_API_URL: https://statsapi.mlb.com/api/v1/schedule


In [12]:
all_start_times = []

for team_abbr, team_id in MLB_TEAM_IDS.items():
    for year in SEASONS:
        try:
            resp = requests.get(MLB_API_URL, params={
                'teamId': team_id,
                'season': year,
                'sportId': 1,
                'gameType': 'R',
            })
            resp.raise_for_status()
            data = resp.json()
            
            for date_entry in data.get('dates', []):
                for game in date_entry.get('games', []):
                    home_team_id = game.get('teams', {}).get('home', {}).get('team', {}).get('id')
                    if home_team_id == team_id:  # Only home games to avoid duplicates
                        all_start_times.append({
                            'game_pk': game['gamePk'],
                            'game_start_utc': game['gameDate'],  # UTC ISO 8601
                        })
        except Exception as e:
            print(f"  WARNING: Failed {team_abbr} {year}: {e}")
        
        time.sleep(0.1)  # Rate limiting
    
    print(f"  {team_abbr}: done ({len([s for s in all_start_times if True])} total so far)")

start_times_df = pd.DataFrame(all_start_times)
start_times_df = start_times_df.drop_duplicates(subset='game_pk')  # Deduplicate
start_times_df['game_pk'] = start_times_df['game_pk'].astype('Int64')

print(f"\nTotal start times retrieved: {len(start_times_df)}")
print(start_times_df.head())

  ARI: done (841 total so far)
  ATL: done (1701 total so far)
  BAL: done (2573 total so far)
  BOS: done (3438 total so far)
  CHC: done (4303 total so far)
  CIN: done (5166 total so far)
  CLE: done (6041 total so far)
  COL: done (6899 total so far)
  CWS: done (7774 total so far)
  DET: done (8656 total so far)
  HOU: done (9500 total so far)
  KC: done (10355 total so far)
  LAA: done (11200 total so far)
  LAD: done (12041 total so far)
  MIA: done (12890 total so far)
  MIL: done (13735 total so far)
  MIN: done (14598 total so far)
  NYM: done (15477 total so far)
  NYY: done (16349 total so far)
  OAK: done (17191 total so far)
  PHI: done (18055 total so far)
  PIT: done (18906 total so far)
  SD: done (19751 total so far)
  SEA: done (20596 total so far)
  SF: done (21438 total so far)
  STL: done (22310 total so far)
  TB: done (23152 total so far)
  TEX: done (23996 total so far)
  TOR: done (24845 total so far)
  WSH: done (25728 total so far)

Total start times retriev

In [13]:
# Merge start times into game data
games = games.merge(start_times_df, on='game_pk', how='left')

n_missing = games['game_start_utc'].isna().sum()
print(f"Games with start times: {len(games) - n_missing} / {len(games)}")
if n_missing > 0:
    print(f"\nWARNING: {n_missing} games missing start times:")
    print(games[games['game_start_utc'].isna()][['game_pk', 'game_date', 'season', 'home_team']].head(20))

Games with start times: 24324 / 24324


In [14]:
# Save checkpoint so you never need to re-run the big Statcast pull above
checkpoint_dir = os.path.join('Final Datasets')
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(checkpoint_dir, 'games_checkpoint.csv')
games.to_csv(checkpoint_path, index=False)

print(f"Checkpoint saved: {os.path.abspath(checkpoint_path)}")
print(f"  Rows: {len(games)}, File size: {os.path.getsize(checkpoint_path) / 1024:.1f} KB")
print(f"\nTo skip the big pull on re-run, restart kernel and run:")
print(f"  Cell 0 (imports) -> Cell 7 (function defs) -> Cell 10 (MLB constants) -> Reload cell below")

Checkpoint saved: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/games_checkpoint.csv
  Rows: 24324, File size: 2260.7 KB

To skip the big pull on re-run, restart kernel and run:
  Cell 0 (imports) -> Cell 7 (function defs) -> Cell 10 (MLB constants) -> Reload cell below


## Reload from Checkpoint (skip here if re-running)

If the kernel was restarted, run these cells first before this one:
1. **Section 0** — imports (cell 2)
2. **Section 2** — function definitions only (cell 7)
3. **Section 3** — MLB constants only (cell 10)

Then run this cell to reload `games` from the checkpoint file.

In [6]:
# Load games from saved file (skips the multi-hour Statcast pull + aggregation)
# Tries checkpoint first, falls back to master_data.csv
checkpoint_path = os.path.join('Final Datasets', 'games_checkpoint.csv')
master_path = os.path.join('Final Datasets', 'master_data.csv')

if os.path.exists(checkpoint_path):
    load_path = checkpoint_path
    print(f"Loading from checkpoint: {checkpoint_path}")
elif os.path.exists(master_path):
    load_path = master_path
    print(f"No checkpoint found. Loading from: {master_path}")
else:
    raise FileNotFoundError("Neither games_checkpoint.csv nor master_data.csv found in Final Datasets/")

games = pd.read_csv(load_path)
games['game_date'] = pd.to_datetime(games['game_date'])

# Restore Int64 types (CSV roundtrip converts to float)
int_cols = ['game_pk', 'home_runs_scored', 'away_runs_scored', 'total_runs',
            'home_runs_hit', 'strikeouts', 'walks', 'hits', 'total_pitches',
            'n_barrels', 'n_bbe', 'season']
for col in int_cols:
    games[col] = pd.to_numeric(games[col], errors='coerce').astype('Int64')

print(f"Loaded: {len(games)} games")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"Columns: {list(games.columns)}")

Loading from checkpoint: Final Datasets/games_checkpoint.csv
Loaded: 24324 games
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Columns: ['game_pk', 'game_date', 'season', 'home_team', 'away_team', 'home_runs_scored', 'away_runs_scored', 'total_runs', 'home_runs_hit', 'strikeouts', 'walks', 'hits', 'total_pitches', 'avg_exit_velocity', 'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio', 'game_start_utc']


## Section 3.5: Recovery Pull for Failed Months

Three monthly Statcast pulls timed out during the initial download: **2016-07**, **2017-05**, and **2025-03**. This cell re-attempts those pulls in **weekly chunks** (smaller payloads) with escalating backoff, aggregates the data, fetches start times, and appends the results to the `games` DataFrame.

In [16]:
# Re-pull the 3 months that timed out, split into WEEKLY chunks to avoid timeouts
# Uses escalating backoff between retries

FAILED_MONTHS_WEEKLY = [
    # July 2016, 07/19/2016 does not pull
    ('2016-07-01', '2016-07-07'),
    ('2016-07-08', '2016-07-14'),
    ('2016-07-15', '2016-07-16'),
    ('2016-07-17', '2016-07-18'),
    ('2016-07-20', '2016-07-21'),
    ('2016-07-22', '2016-07-28'),
    ('2016-07-29', '2016-07-31'),
    # May 2017, 05/06/2017 does not pull
    ('2017-05-01', '2017-05-02'),
    ('2017-05-03', '2017-05-04'),
    ('2017-05-05', '2017-05-05'),
    ('2017-05-07', '2017-05-07'),
    ('2017-05-08', '2017-05-14'),
    ('2017-05-15', '2017-05-21'),
    ('2017-05-22', '2017-05-28'),
    ('2017-05-29', '2017-05-31'),
    # March 2025, 03/28/2025 does not pull
    ('2025-03-01', '2025-03-07'),
    ('2025-03-08', '2025-03-14'),
    ('2025-03-15', '2025-03-16'),
    ('2025-03-17', '2025-03-18'),
    ('2025-03-19', '2025-03-21'),
    ('2025-03-22', '2025-03-23'),
    ('2025-03-24', '2025-03-25'),
    ('2025-03-26', '2025-03-26'),
    ('2025-03-27', '2025-03-27'),
    ('2025-03-29', '2025-03-31'),
]

MAX_RETRIES = 8
RETRY_BASE_WAIT = 30  # seconds; escalates with each attempt

# ── Step 1: Pull pitch-level data in weekly chunks ──────────────────────────
recovery_pitches_list = []

for start_dt, end_dt in FAILED_MONTHS_WEEKLY:
    print(f"Pulling {start_dt} to {end_dt}...")
    success = False
    for attempt in range(MAX_RETRIES):
        try:
            df = statcast(start_dt=start_dt, end_dt=end_dt, verbose=False)
            if df is not None and len(df) > 0:
                recovery_pitches_list.append(df)
                print(f"  SUCCESS: {len(df):,} pitches")
            else:
                print(f"  No data for {start_dt} to {end_dt}")
            success = True
            break
        except Exception as e:
            wait = RETRY_BASE_WAIT * (attempt + 1)
            if attempt < MAX_RETRIES - 1:
                print(f"  Attempt {attempt + 1}/{MAX_RETRIES} failed: {e}")
                print(f"  Waiting {wait}s before retry...")
                time.sleep(wait)
            else:
                print(f"  FAILED after {MAX_RETRIES} attempts: {e}")

    if not success:
        print(f"  >> SKIPPED {start_dt} to {end_dt} — all retries exhausted")

if not recovery_pitches_list:
    print("\nNo recovery data retrieved. Skipping remaining steps.")
else:
    recovery_pitches_raw = pd.concat(recovery_pitches_list, ignore_index=True)
    print(f"\nTotal recovery pitches (raw): {len(recovery_pitches_raw):,}")

    # ── Step 2: Filter to regular season ────────────────────────────────────
    recovery_pitches = recovery_pitches_raw[
        recovery_pitches_raw['game_type'] == 'R'
    ].copy()
    recovery_pitches['game_date'] = pd.to_datetime(recovery_pitches['game_date'])
    recovery_pitches['season'] = recovery_pitches['game_date'].dt.year

    print(f"Recovery pitches (regular season): {len(recovery_pitches):,}")
    print(f"Recovery games: {recovery_pitches['game_pk'].nunique()}")

    if len(recovery_pitches) == 0:
        print("\nNo regular season games found in recovery data. "
              "This is expected for March 2025 (season starts late March).")
    else:
        # ── Step 3: Aggregate to game level ─────────────────────────────────
        print("\nAggregating recovery games...")
        recovery_games = (
            recovery_pitches
            .groupby('game_pk')
            .apply(aggregate_game)
            .reset_index(drop=True)
        )
        recovery_games['game_date'] = pd.to_datetime(recovery_games['game_date'])

        # Same integer casting as Section 2
        int_cols = [
            'game_pk', 'home_runs_scored', 'away_runs_scored', 'total_runs',
            'home_runs_hit', 'strikeouts', 'walks', 'hits', 'total_pitches',
            'n_barrels', 'n_bbe', 'season',
        ]
        for col in int_cols:
            recovery_games[col] = pd.to_numeric(
                recovery_games[col], errors='coerce'
            ).astype('Int64')

        recovery_games['avg_exit_velocity'] = recovery_games['avg_exit_velocity'].round(1)
        recovery_games['barrel_rate'] = recovery_games['barrel_rate'].round(4)
        recovery_games['hr_h_ratio'] = recovery_games['hr_h_ratio'].round(4)

        print(f"Recovery games aggregated: {len(recovery_games)}")

        # ── Step 4: Get start times for recovery games ──────────────────────
        recovery_team_seasons = (
            recovery_games[['home_team', 'season']]
            .drop_duplicates()
            .values.tolist()
        )

        print(f"\nFetching start times for {len(recovery_team_seasons)} "
              f"team-season combinations...")

        recovery_start_times = []
        for team_abbr, season in recovery_team_seasons:
            team_id = MLB_TEAM_IDS.get(team_abbr)
            if team_id is None:
                print(f"  WARNING: Unknown team '{team_abbr}', skipping")
                continue
            try:
                resp = requests.get(MLB_API_URL, params={
                    'teamId': team_id,
                    'season': int(season),
                    'sportId': 1,
                    'gameType': 'R',
                })
                resp.raise_for_status()
                data = resp.json()

                for date_entry in data.get('dates', []):
                    for game in date_entry.get('games', []):
                        home_id = (
                            game.get('teams', {})
                            .get('home', {})
                            .get('team', {})
                            .get('id')
                        )
                        if home_id == team_id:
                            recovery_start_times.append({
                                'game_pk': game['gamePk'],
                                'game_start_utc': game['gameDate'],
                            })
            except Exception as e:
                print(f"  WARNING: Failed {team_abbr} {season}: {e}")

            time.sleep(0.1)

        recovery_start_df = pd.DataFrame(recovery_start_times)
        if len(recovery_start_df) > 0:
            recovery_start_df = recovery_start_df.drop_duplicates(subset='game_pk')
            recovery_start_df['game_pk'] = recovery_start_df['game_pk'].astype('Int64')

        # ── Step 5: Merge start times into recovery games ───────────────────
        if len(recovery_start_df) > 0:
            recovery_games = recovery_games.merge(
                recovery_start_df, on='game_pk', how='left'
            )
        else:
            recovery_games['game_start_utc'] = pd.NaT

        n_missing_st = recovery_games['game_start_utc'].isna().sum()
        print(f"Recovery games with start times: "
              f"{len(recovery_games) - n_missing_st} / {len(recovery_games)}")

        # ── Step 6: Append to main games DataFrame ──────────────────────────
        existing_count = len(games)

        # Only add games not already in the DataFrame
        existing_pks = set(games['game_pk'].dropna().astype(int))
        new_mask = ~recovery_games['game_pk'].isin(existing_pks)
        truly_new = recovery_games[new_mask]
        n_skipped = len(recovery_games) - len(truly_new)

        if n_skipped > 0:
            print(f"\nSkipped {n_skipped} games already present in `games`")

        if len(truly_new) > 0:
            games = pd.concat([games, truly_new], ignore_index=True)
            games = games.drop_duplicates(subset='game_pk', keep='first')
            games = games.sort_values(['game_date', 'game_pk']).reset_index(drop=True)

            print(f"\nGames before recovery: {existing_count}")
            print(f"New games added:       {len(truly_new)}")
            print(f"Games after recovery:  {len(games)}")
        else:
            print(f"\nNo new games to add "
                  f"(all {len(recovery_games)} already existed)")

    print(f"\n{'='*60}")
    print("RECOVERY COMPLETE")
    print(f"{'='*60}")
    print(f"Total games in `games`: {len(games)}")
    print(f"\nGames per season (updated):")
    print(games.groupby('season')['game_pk'].count())

Pulling 2016-07-01 to 2016-07-07...


100%|██████████| 7/7 [00:10<00:00,  1.45s/it]


  SUCCESS: 30,407 pitches
Pulling 2016-07-08 to 2016-07-14...


100%|██████████| 7/7 [00:10<00:00,  1.48s/it]


  SUCCESS: 13,322 pitches
Pulling 2016-07-15 to 2016-07-16...


...

  No data for 2025-03-26 to 2025-03-26
Pulling 2025-03-27 to 2025-03-27...


100%|██████████| 1/1 [00:00<00:00,  1.49it/s]


  SUCCESS: 4,343 pitches
Pulling 2025-03-29 to 2025-03-31...


100%|██████████| 3/3 [00:02<00:00,  1.10it/s]


  SUCCESS: 12,298 pitches

Total recovery pitches (raw): 291,684
Recovery pitches (regular season): 247,662
Recovery games: 831

Aggregating recovery games...
Recovery games aggregated: 831

Fetching start times for 80 team-season combinations...
Recovery games with start times: 782 / 831

Games before recovery: 24324
New games added:       831
Games after recovery:  25155

RECOVERY COMPLETE
Total games in `games`: 25155

Games per season (updated):
season
2015    2429
2016    2414
2017    2415
2018    2431
2019    2429
2020     898
2021    2429
2022    2430
2023    2430
2024    2429
2025    2421
Name: game_pk, dtype: Int64


## Section 4: Validation

Verify row counts, check for nulls, and sanity-check summary statistics.

In [17]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Overall shape
print(f"\n--- Dataset Shape ---")
print(f"  Rows: {len(games)}, Columns: {len(games.columns)}")
print(f"  Unique teams: {games['home_team'].nunique()}")
print(f"  Seasons: {sorted(games['season'].unique())}")

# 2. Games per season (league-wide)
print(f"\n--- Games per Season (all teams) ---")
season_counts = games.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (750, 1050)  # COVID: ~30 teams x 30 home games
    else:
        expected = (2300, 2600)  # Normal: ~30 teams x 81 home games
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(games)} games")

# 3. Duplicate check
n_dupes = games['game_pk'].duplicated().sum()
print(f"\n--- Duplicate game_pk: {n_dupes} {'[OK]' if n_dupes == 0 else '[WARNING]'} ---")

# 4. Null check
print(f"\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate', 'game_start_utc']
for col in key_cols:
    n_null = games[col].isna().sum()
    pct = 100 * n_null / len(games)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 5. Sanity checks
print(f"\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', games['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', games['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', games['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', games['walks'].mean(), '~6-7'),
    ('Avg exit velocity', games['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', games['barrel_rate'].mean(), '~0.06-0.08'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 6. Games per team (spot check)
print(f"\n--- Games per Team (total across all seasons) ---")
team_counts = games.groupby('home_team').size().sort_values(ascending=False)
print(team_counts.to_string())

VALIDATION REPORT

--- Dataset Shape ---
  Rows: 25155, Columns: 19
  Unique teams: 30
  Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

--- Games per Season (all teams) ---
  2015: 2429 games [OK] (expected 2300-2600)
  2016: 2414 games [OK] (expected 2300-2600)
  2017: 2415 games [OK] (expected 2300-2600)
  2018: 2431 games [OK] (expected 2300-2600)
  2019: 2429 games [OK] (expected 2300-2600)
  2020: 898 games [OK] (expected 750-1050)
  2021: 2429 games [OK] (expected 2300-2600)
  2022: 2430 games [OK] (expected 2300-2600)
  2023: 2430 games [OK] (expected 2300-2600)
  2024: 2429 games [OK] (expected 2300-2600)
  2025: 2421 games [OK] (expected 2300-2600)
  TOTAL: 25155 games

--- Duplicate game_pk: 0 [OK] ---

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 38 nulls (0.2%) [OK]
  barrel_rate:

In [18]:
# Full summary statistics
print("--- Summary Statistics ---")
print(games.describe().T[['mean', 'std', 'min', 'max']].to_string())

--- Summary Statistics ---
                                            mean            std                  min                  max
game_pk                            600020.099543  119048.435625             413649.0             778564.0
game_date          2020-07-04 11:01:41.753130496            NaN  2015-04-05 00:00:00  2025-09-28 00:00:00
season                               2020.002504        3.25584               2015.0               2025.0
home_runs_scored                        4.544107       3.141136                  0.0                 29.0
away_runs_scored                        4.452276       3.228266                  0.0                 28.0
total_runs                              8.996382       4.526971                  0.0                 38.0
home_runs_hit                           2.358736        1.67896                  0.0                 13.0
strikeouts                             16.783264       4.328101                  3.0                 48.0
walks              

## Section 5: Save to CSV

In [19]:
# Final column order
final_columns = [
    'game_pk', 'game_date', 'season', 'home_team', 'away_team', 'game_start_utc',
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
]

master_data = games[final_columns].copy()
master_data = master_data.sort_values(['game_date', 'game_pk']).reset_index(drop=True)

# Save
output_dir = os.path.join('Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'master_data.csv')
master_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(master_data)}, Columns: {len(master_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == master_data.shape, f"Shape mismatch: {verify.shape} vs {master_data.shape}"
print("\nSave & reload verification: PASSED")

print(f"\n--- First 10 Rows ---")
master_data.head(10)

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/master_data.csv
File size: 2337.0 KB
Rows: 25155, Columns: 19

Save & reload verification: PASSED

--- First 10 Rows ---


,game_pk,game_date,season,home_team,away_team,game_start_utc,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio
0,413661,2015-04-05,2015,CHC,STL,2015-04-06T00:05:00Z,0,3,3,0,23,6,15,314,85.3,1,45,0.0222,0.0000
1,413649,2015-04-06,2015,SEA,LAA,2015-04-06T20:10:00Z,4,1,5,2,15,3,14,253,89.9,1,46,0.0217,0.1429
2,413650,2015-04-06,2015,MIA,ATL,2015-04-06T20:10:00Z,1,2,3,0,14,2,14,243,87.1,1,51,0.0196,0.0000
3,413651,2015-04-06,2015,TB,BAL,2015-04-06T19:10:00Z,2,6,8,4,19,5,16,295,87.1,3,47,0.0638,0.2500
4,413652,2015-04-06,2015,PHI,BOS,2015-04-06T19:05:00Z,0,8,8,5,18,9,12,302,87.7,1,48,0.0208,0.4167
5,413653,2015-04-06,2015,KC,CWS,2015-04-06T20:10:00Z,10,1,11,3,6,7,18,257,NaN,0,57,0.0000,0.1667
6,413654,2015-04-06,2015,HOU,CLE,2015-04-06T23:10:00Z,2,0,2,0,14,5,6,230,82.0,2,41,0.0488,0.0000
7,413655,2015-04-06,2015,MIL,COL,2015-04-06T18:10:00Z,0,10,10,2,15,1,24,284,87.7,0,61,0.0000,0.0833
8,413656,2015-04-06,2015,DET,MIN,2015-04-06T17:08:00Z,4,0,4,2,12,2,15,229,92.3,3,51,0.0588,0.1333
9,413657,2015-04-06,2015,WSH,NYM,2015-04-06T20:05:00Z,1,3,4,1,19,4,8,245,86.8,2,46,0.0435,0.1250
